# Flux
[Flux](https://flux-framework.readthedocs.io) is a modern HPC resource manager and job scheduler developed at Lawrence Livermore National Laboratory (LLNL). Like SLURM, it manages queues and dispatches jobs to compute nodes — but it is designed specifically for **hierarchical scheduling** and **many-task workflows**, making it an excellent match for the kind of Python-function-level parallelism that `executorlib` uses.

**Why Flux in this tutorial?**
LANL's HPC systems run SLURM as the primary scheduler. Flux is used as a *second-level scheduler* that runs *inside* a SLURM job: you obtain an allocation with `sbatch`/`salloc`, start a Flux instance with `srun flux start`, and then use Flux commands to subdivide that allocation into many smaller tasks. This two-level approach lets you submit thousands of lightweight sub-jobs without overwhelming the SLURM scheduler with individual batch scripts.

In this notebook's Jupyter environment a Flux instance is already running, so you can use Flux commands directly without first obtaining a SLURM allocation.

The basic command mapping is:

| SLURM | Flux | Purpose |
|---|---|---|
| `sbatch` | `flux submit` | submit a non-blocking job |
| `srun` | `flux run` | launch a blocking interactive job |
| `squeue` | `flux jobs` | inspect the job queue |
| `sinfo` | `flux resource list` | inspect available resources |

Additional Flux references:
* [Integrate Flux with SLURM](https://flux-framework.readthedocs.io/projects/flux-core/en/stable/guide/start.html#starting-with-slurm)
* [Flux cheat sheet](https://flux-framework.org/cheat-sheet/)
* [Flux Learning Guide](https://flux-framework.readthedocs.io/en/latest/guides/learning_guide.html)

## Basic Flux Commands
### `flux start`
`flux start` launches a new Flux scheduler instance inside an **existing** allocation. This is how Flux is used on top of SLURM at LANL:

```bash
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --error=error.out
#SBATCH --job-name=test
#SBATCH --chdir=/u/janj/slurmtest
#SBATCH --get-user-env=L
#SBATCH --partition=s.cmfe
#SBATCH --time=60
#SBATCH --ntasks=8

srun flux start workflow.sh
```

Once `flux start` is running, the script `workflow.sh` can use all Flux sub-commands to distribute work across the eight tasks that SLURM allocated:

```bash
flux run -n 4 python script.py &
flux run -n 4 python script.py &
wait
```

Inside the Jupyter environment used for this tutorial a Flux instance is already active, so the following commands can be run directly without a surrounding `sbatch` / `srun flux start` wrapper.


### `flux resource list`

Inspect the available computing resources managed by the current Flux instance:
```bash
flux resource list
```

Example output:
```
     STATE NNODES NCORES NGPUS NODELIST
      free      1     24     0 jupyter-pyiron-workshop-torlib-tutorial-fiekunpl
 allocated      0      0     0
      down      0      0     0
```

**Column meanings:**
- `STATE`: current state of the resources — `free` (idle and available), `allocated` (in use by a running job), or `down` (unavailable, e.g. hardware fault)
- `NNODES`: number of compute nodes
- `NCORES`: total CPU cores across those nodes
- `NGPUS`: total GPUs across those nodes
- `NODELIST`: hostname(s) of the nodes

Run this command before submitting work to confirm that the Flux instance has access to compute resources and that enough cores are free for your jobs.


In [1]:
!flux resource list

     STATE NNODES NCORES NGPUS NODELIST
      free      1     24     0 jupyter-pyiron-workshop-torlib-tutorial-bpficiu6
 allocated      0      0     0 
      down      0      0     0 


### `flux jobs`

Inspect the Flux job queue to monitor submitted and running jobs.

List currently active (running or pending) jobs:
```bash
flux jobs
```

Show **all** jobs including completed and failed ones:
```bash
flux jobs -a
```

Example output columns:
```
       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
  ƒ76Mhphu   jovyan   python      R      4      1   0:02:13 jupyter-...
```

- `JOBID`: a Flux-specific job identifier (prefixed with `ƒ`); use this to attach to or cancel a job
- `ST`: job state — `R` (running), `PD` (pending), `CD` (completed), `F` (failed)
- `NTASKS`/`NNODES`: how many tasks and nodes were requested
- `TIME`: elapsed wall-clock time

To attach to a running or completed job and see its output:
```bash
flux job attach <JOBID>
```


In [2]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO


### `flux submit`
Submit a **non-blocking** job to the Flux scheduler. The command returns immediately with a job ID; the job runs asynchronously in the background.

```bash
flux submit --nodes=1 --ntasks=4 python script.py
```

Returns a Flux job ID (e.g. `ƒ76Mhphu`) which you can use to track the job:

```bash
flux jobs -a            # see status
flux job attach ƒ76Mhphu  # retrieve output once finished
```

Common `flux submit` flags:

| Flag | Meaning |
|---|---|
| `--nodes=N` | request N compute nodes |
| `--ntasks=N` | request N parallel tasks |
| `-o pmi=pmix` | enable PMIx process-management interface for MPI |
| `--output=<file>` | redirect stdout to a file |
| `--error=<file>` | redirect stderr to a file |

> **Analogy with SLURM:** `flux submit` ≈ `sbatch` — both are non-blocking and return a job ID.


In [3]:
lines = """\
from mpi4py import MPI
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()
print(rank, size)
"""

with open("script.py", "w") as f:
    f.writelines(lines)

In [4]:
!flux submit -o pmi=pmix --nodes=1 --ntasks=4 python script.py

ƒ76Mhphu


In [5]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒ76Mhphu jovyan   python     CD      4      1   1.836s jupyter-pyiron-workshop-torlib-tutorial-bpficiu6


In [6]:
!flux job attach ƒ76Mhphu

1 4
0 4
3 4
2 4


### `flux run`
Launch a **blocking** interactive job. The command waits until the job finishes and then returns, printing output directly to the terminal (or notebook cell).

```bash
flux run -n 4 python script.py
```

Common `flux run` flags mirror those of `flux submit`; the most important for MPI work is `-o pmi=pmix` which enables the PMIx bootstrap used by modern OpenMPI builds:

```bash
flux run -o pmi=pmix -n 4 python script.py
```

> **Analogy with SLURM:** `flux run` ≈ `srun` — both are blocking and print output in real time.

Use `flux run` when you need to see output immediately (interactive debugging, tutorials). Use `flux submit` when submitting many jobs that should run in parallel without blocking your session.


In [7]:
!flux run -o pmi=pmix -n 4 python script.py

3 4
1 4
0 4
2 4


In [8]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒBftPC7y jovyan   python     CD      4      1   1.837s jupyter-pyiron-workshop-torlib-tutorial-bpficiu6
    ƒ76Mhphu jovyan   python     CD      4      1   1.836s jupyter-pyiron-workshop-torlib-tutorial-bpficiu6


## mpi4py Examples with Flux

The same two strategies for distributing MPI work that were shown for SLURM in the previous notebook apply equally with Flux.

### Multiple `flux run` Calls

Launch two independent four-task MPI jobs concurrently within the same Flux allocation:

```bash
flux run -o pmi=pmix -n 4 python script.py &
flux run -o pmi=pmix -n 4 python script.py &
wait
```

The `-o pmi=pmix` option enables the PMIx process-management interface, which is required for `mpi4py` to correctly discover all MPI ranks. Each `flux run` spawns a separate MPI job of size 4; the two jobs run simultaneously and each sees its own `MPI.COMM_WORLD`.


In [9]:
!flux run -o pmi=pmix -n 4 python script.py & flux run -o pmi=pmix -n 4 python script.py & wait

2 4
1 4
0 4
3 4
2 4
0 4
3 4
1 4


### One `flux run` and Split the MPI Communicator

As an alternative to launching two separate `flux run` jobs, launch a single eight-task MPI job and divide the communicator inside Python. Save the following as `mpisplit.py`:

```python
from mpi4py import MPI

def task(comm):
    rank = comm.Get_rank()
    size = comm.Get_size()
    print(rank, size)

world = MPI.COMM_WORLD
rank = world.Get_rank()
color = 0 if rank < 4 else 1
subcomm = world.Split(color=color, key=rank)
if color == 0:
    # ranks 0–3 run task A
    task(subcomm)
else:
    # ranks 4–7 run task B
    task(subcomm)
```

Run with:
```bash
flux run -o pmi=pmix -n 8 python mpisplit.py
```

This launches one MPI job of size 8 and divides it into two logical communicators of size 4. See the SLURM notebook for a detailed discussion of when to prefer this approach over multiple `flux run` calls.


In [10]:
mpi_lines = """\
from mpi4py import MPI

def task(comm):
    rank = comm.Get_rank()
    size = comm.Get_size()
    print(rank, size)

world = MPI.COMM_WORLD
rank = world.Get_rank()
color = 0 if rank < 4 else 1
subcomm = world.Split(color=color, key=rank)
if color == 0:
    # ranks 0–3 run task A
    task(subcomm)
else:
    # ranks 4–7 run task B
    task(subcomm)
"""

with open("mpisplit.py", "w") as f:
    f.writelines(mpi_lines)

In [11]:
!flux run -o pmi=pmix -n 8 python mpisplit.py

3 4
2 4
1 4
0 4
3 4
2 4
1 4
0 4


## Flux Python Interface

Beyond the command-line tools (`flux submit`, `flux run`), Flux provides a native **Python API** in the `flux` package. This allows Python scripts to submit jobs programmatically — useful when the number of jobs or their parameters are determined at runtime:

```python
import flux.job

jobid = flux.job.submit(
    flux.Flux(),
    jobspec=flux.job.JobspecV1.from_command(
        command=["hostname"],
        num_tasks=4,
    ),
)

print(jobid)
```

`flux.Flux()` opens a connection to the running Flux broker. `JobspecV1.from_command()` constructs a job specification (equivalent to the flags passed on the CLI). `flux.job.submit()` submits the job non-blocking and returns a numeric job ID.

> **Why does this matter for executorlib?**
> `executorlib`'s `FluxClusterExecutor` and `FluxJobExecutor` use exactly this Python API > internally to submit individual Python functions as Flux jobs. Understanding the Python > interface helps demystify what executorlib is doing under the hood.


In [12]:
import flux.job

jobid = flux.job.submit(
    flux.Flux(),
    jobspec=flux.job.JobspecV1.from_command(
        command=["hostname"],
        num_tasks=4,
    ),
)

print(jobid)

ƒMhJMY7R


In [13]:
!flux job attach $jobid

jupyter-pyiron-workshop-torlib-tutorial-bpficiu6
jupyter-pyiron-workshop-torlib-tutorial-bpficiu6
jupyter-pyiron-workshop-torlib-tutorial-bpficiu6
jupyter-pyiron-workshop-torlib-tutorial-bpficiu6
